# About
- A `TinyLM`, autoregressive model for a simple storyteller.
- An autoregressive model, i.e. generating a story.

# Setup

In [22]:
from pprint import pprint
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from transformers import set_seed
from transformers.utils import logging

# Device.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_bf16 = (device.type == 'cuda') and (torch.cuda.is_bf16_supported())

# Seed.
seed = 42
set_seed(seed)

# Suppress loggings.
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
logging.set_verbosity_error()
logging.disable_progress_bar()

# 1. Data Preparation

## Dataset

- A short story dataset, synthetically generated by gpt-3.5 and gpt-4.
- A large size: train (2,119,719) and valid (21,990).
- Ideal for testing whether a small LM could learn grammar, basic reasoning, storytelling, and so on.
- Intentionally uses simple and small vocabulary.

In [23]:
from datasets import load_dataset, DatasetDict

data = load_dataset('roneneldan/TinyStories')

ds = DatasetDict({
    'train': data['train'],
    'test': data['validation'],
})

pprint(ds['train'][0])

{'text': 'One day, a little girl named Lily found a needle in her room. She '
         'knew it was difficult to play with it because it was sharp. Lily '
         'wanted to share the needle with her mom, so she could sew a button '
         'on her shirt.\n'
         '\n'
         'Lily went to her mom and said, "Mom, I found this needle. Can you '
         'share it with me and sew my shirt?" Her mom smiled and said, "Yes, '
         'Lily, we can share the needle and fix your shirt."\n'
         '\n'
         "Together, they shared the needle and sewed the button on Lily's "
         'shirt. It was not difficult for them because they were sharing and '
         'helping each other. After they finished, Lily thanked her mom for '
         'sharing the needle and fixing her shirt. They both felt happy '
         'because they had shared and worked together.'}


## Tokenizer

- Normalizer: normalize a sentence.
  - Modern LLMs commonly avoid aggresive BERT-style normalization, e.g. lowercasing.
  - We skip normalizer for TinyLM.
- Pre-tokenizer: sentence -> words.
  - Split by whitespace.
- Model: subword model, byte-level BPE.
- Post-processor: appends special tokens.
  - `[UNK]`: unknown tokens.
  - `[CLS]`: beginning of sequence.
  - `[SEQ]`: end of sequence.
  - `[PAD]`: padding.

### Train

In [24]:
from tqdm import tqdm
from tokenizers import(
    Tokenizer,
    models,
    pre_tokenizers,
    processors,
    trainers,
    decoders,
)
from transformers import PreTrainedTokenizerFast


# Tokenizer, with model = BPE.
tokenizer = Tokenizer(models.BPE())

# Pre-tokenizer.
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(
    add_prefix_space=False,
)

# Decoder.
tokenizer.decoder = decoders.ByteLevel()

# Train.
n_train_tokenize = 50_000
train_samples = ds['train'].select(range(n_train_tokenize))
train_texts = tqdm(
    train_samples["text"],
    total=len(train_samples),
    desc="Training tokenizer"
)
trainer = trainers.BpeTrainer(
    vocab_size=8_000,
    min_frequency=2,
    special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),   # add 256 unicode chars in the vocab.
    max_token_length=256,
    show_progress=True,
)
tokenizer.train_from_iterator(
    train_texts,
    trainer=trainer,
    length=n_train_tokenize,
)

# Post-processor.
cls_id = tokenizer.token_to_id("[CLS]")
sep_id = tokenizer.token_to_id("[SEP]")
tokenizer.post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    special_tokens=[
        ("[CLS]", cls_id),
        ("[SEP]", sep_id),
    ],
)

Training tokenizer: 100%|██████████| 50000/50000 [00:01<00:00, 26373.95it/s]


### Wrapper

In [25]:
# Wrapper.
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="[UNK]",
    cls_token="[CLS]",
    sep_token="[SEP]",
    pad_token="[PAD]",
    model_max_length=256,
)

### Tokenize

In [26]:
# Sample.
ids = tokenizer('This is Seoul')['input_ids']
tokens = tokenizer.convert_ids_to_tokens(ids)
print(tokens)

# Dataset.
n_train = 10_000       # train size for LLM.
train_ds = ds['train'].select(range(n_train))
def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=256,
    )
tokenized = train_ds.map(tokenize, batched=True, remove_columns=['text'])

['[CLS]', 'This', 'Ġis', 'ĠS', 'e', 'ou', 'l', '[SEP]']


In [27]:
l = [len(x) for x in tokenized['input_ids']]
s = pd.Series(l)
s.describe()

count    10000.000000
mean       193.486900
std         41.431566
min         55.000000
25%        163.000000
50%        190.000000
75%        226.000000
max        256.000000
dtype: float64

## Data Collator

In [28]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # clm -> creates `labels` from `input_ids`, and paddings -> -100.
)

# 2. Model

```markdown
TinyEmbedding
      ↓
TinyBlock × N
  ├─ Attention → residual → norm
  └─ FFN       → residual → norm
      ↓
LM head
      ↓
(B, L, V)
```

## Embedding

In [29]:
class TinyEmbedding(nn.Module):
    def __init__(self, d, V, T, p):
        super().__init__()

        self.d = d      # embedding dimension.
        self.V = V      # vocab size.
        self.T = T      # max token size of the transformer.
        self.p = p      # dropout ratio.

        self.word_embedding = nn.Embedding(V, d)
        self.position_embedding = nn.Embedding(T, d)

        self.emb_norm = nn.LayerNorm(d)
        self.dropout = nn.Dropout(p=p)

    def forward(self, x):
        B, L = x.shape

        word_emb = self.word_embedding(x)
        positions = torch.arange(L, device=x.device)
        pos_emb = self.position_embedding(positions)
        x = word_emb + pos_emb
        x = self.emb_norm(x)
        x = self.dropout(x)

        return x

## Attention

In [30]:
class TinyAttention(nn.Module):
    def __init__(self, d, V, T, p):
        super().__init__()

        self.d = d      # embedding dimension.
        self.V = V      # vocab size.
        self.T = T      # max token size of the transformer.
        self.p = p      # dropout ratio.

        self.wq = nn.Linear(d, d)
        self.wk = nn.Linear(d, d)
        self.wv = nn.Linear(d, d)
        self.attn_dropout = nn.Dropout(p=p)
        self.attn_out = nn.Linear(d, d)
        self.attn_norm = nn.LayerNorm(d)

    def forward(self, x):
        B, L, d = x.shape

        # QKV projection.
        q = self.wq(x)      # (B, L, d)
        k = self.wk(x)
        v = self.wv(x)

        # Attention scores.
        scores = q @ k.transpose(-2, -1)        # (B, L, L)
        scores = scores / (self.d ** 0.5)

        # Causal mask.
        mask = torch.triu(
            torch.ones(L, L, device=x.device, dtype=torch.bool),
            diagonal=1,
        )
        scores = scores.masked_fill(mask, float("-inf"))

        # Weights.
        weights = torch.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        # Contexts.
        contexts = weights @ v
        contexts = self.attn_out(contexts)

        # Residual.
        residual = x + contexts
        residual = self.attn_norm(residual)

        return residual

## FFN

In [31]:
class TinyFFN(nn.Module):
    def __init__(self, d, d_ffn):
        super().__init__()

        self.d = d              # embedding dimension.
        self.d_ffn = d_ffn      # ffn dimension.

        self.hidden1 = nn.Linear(d, d_ffn)
        self.hidden2 = nn.Linear(d_ffn, d)
        self.hidden_activation = nn.GELU()

    def forward(self, x):
        x = self.hidden1(x)
        x = self.hidden_activation(x)
        x = self.hidden2(x)

        return x

## Block

```markdown
x
│
├───────────────┐
↓               │
Attention       │ residual
↓               │
+ ←─────────────┘
↓
LayerNorm
│
├───────────────┐
↓               │
FFN             │ residual
↓               │
+ ←─────────────┘
↓
LayerNorm
↓
output
```

In [32]:
class TinyBlock(nn.Module):
    def __init__(self, d, V, T, p=0.1, d_ffn=256):
        super().__init__()

        self.d = d                  # embedding dimension.
        self.V = V                  # vocab size.
        self.T = T                  # max token size of the transformer.
        self.p = p                  # dropout ratio.
        self.d_ffn = d_ffn          # ffn dimension.

        # Attention.
        self.attention = TinyAttention(d, V, T, p)
        self.attn_norm = nn.LayerNorm(d)

        # FFN.
        self.ffn = TinyFFN(d, d_ffn)
        self.ffn_norm = nn.LayerNorm(d)

    def forward(self, x):
        # Attention + residual.
        x = x + self.attention(x)
        x = self.attn_norm(x)

        # FFN + residual.
        x = x + self.ffn(x)
        x = self.ffn_norm(x)

        return x

## Transformer

In [33]:
class TinyLM(nn.Module):
    def __init__(self, d, V, T, p=0.1, d_ffn=256, n_blocks=1):
        super().__init__()

        self.d = d                  # embedding dimension.
        self.V = V                  # vocab size.
        self.T = T                  # max token size of the transformer.
        self.p = p                  # dropout ratio.
        self.d_ffn = d_ffn          # ffn dimension.
        self.n_blocks = n_blocks    # number of transformer blocks.

        # Embedding.
        self.embedding = TinyEmbedding(d, V, T, p)

        # Transformer blocks.
        self.blocks = nn.ModuleList([
            TinyBlock(d, V, T, p, d_ffn)
            for _ in range(n_blocks)
        ])

        # Language-model head.
        self.lm_head = nn.Linear(d, V)

    def forward(self, input_ids, labels=None, attention_mask=None):
        # Embedding.
        x = self.embedding(input_ids)          # (B, L, d)

        # Transformer blocks.
        for block in self.blocks:
            x = block(x)               # (B, L, d)

        # Language-model head.
        logits = self.lm_head(x)       # (B, L, V)

        # Loss.
        loss = None

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = F.cross_entropy(
                shift_logits.view(-1, self.V),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return {
            "loss": loss,
            "logits": logits,
        }

model = TinyLM(
    d=128,
    V=tokenizer.vocab_size,
    T=tokenizer.model_max_length,
    p=0.1,
    d_ffn=256,
    n_blocks=1,
).to(device)

> Note) A custom LM using `huggingface` usually calculates and returns a **loss** in `forward()` too.

# 3. Training

In [34]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../tmp/tinylm",
    num_train_epochs=5,
    per_device_train_batch_size=64,
    learning_rate=3e-4,
    weight_decay=1e-2,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=collator,
    processing_class=tokenizer,
)

training_results = trainer.train()

pprint(training_results.metrics)

{'loss': '7.302', 'grad_norm': '0.2369', 'learning_rate': '0.0002622', 'epoch': '0.6369'}
{'loss': '5.66', 'grad_norm': '0.2015', 'learning_rate': '0.0002239', 'epoch': '1.274'}
{'loss': '5.29', 'grad_norm': '0.19', 'learning_rate': '0.0001857', 'epoch': '1.911'}
{'loss': '5.052', 'grad_norm': '0.1725', 'learning_rate': '0.0001475', 'epoch': '2.548'}
{'loss': '4.904', 'grad_norm': '0.1791', 'learning_rate': '0.0001093', 'epoch': '3.185'}
{'loss': '4.811', 'grad_norm': '0.1835', 'learning_rate': '7.108e-05', 'epoch': '3.822'}
{'loss': '4.763', 'grad_norm': '0.1526', 'learning_rate': '3.287e-05', 'epoch': '4.459'}
{'train_runtime': '15.04', 'train_samples_per_second': '3325', 'train_steps_per_second': '52.2', 'train_loss': '5.326', 'epoch': '5'}
{'epoch': 5.0,
 'train_loss': 5.3261544197228305,
 'train_runtime': 15.0383,
 'train_samples_per_second': 3324.854,
 'train_steps_per_second': 52.2}


# 4. Generation

In [35]:
def generate(model, tokenizer, prompt, max_new_tokens=50):
    model.eval()

    # Tokenize prompt without [SEP].
    input_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )

    # Add [CLS] at the beginning.
    cls = torch.tensor([[tokenizer.cls_token_id]])
    input_ids = torch.cat([cls, input_ids], dim=1)
    input_ids = input_ids.to(next(model.parameters()).device)

    # Generating tokens.
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=input_ids)
            logits = outputs["logits"]

            # Predict next token.
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)

            # Append.
            input_ids = torch.cat([input_ids, next_token], dim=1)

            # Stop at [SEP].
            if next_token.item() == tokenizer.sep_token_id:
                break

    # Decode.
    text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

    return text

# Generation.
text = generate(
    model,
    tokenizer,
    prompt="Once upon a time",
    max_new_tokens=40,
)
pprint(text)

('Once upon a time, there was a little girl named Lily. She loved to play with '
 'the park.\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n'
 '\n')


# 5. Save and Load

## Save

In [36]:
path = "../tmp/tinylm"

# Config.
config = {
    "d": model.d,
    "V": model.V,
    "T": model.T,
    "p": model.p,
    "d_ffn": model.d_ffn,
    "n_blocks": model.n_blocks,
}

torch.save({
        "model_state_dict": model.state_dict(),     # save weights.
        "config": config    # save config.
    },
    f"{path}/tinylm.pt",
)
tokenizer.save_pretrained(path)     # save tokenizer.

('../tmp/tinylm\\tokenizer_config.json', '../tmp/tinylm\\tokenizer.json')

## Load

In [38]:
from transformers import PreTrainedTokenizerFast

path = "../tmp/tinylm"

# Tokenizer.
tokenizer = PreTrainedTokenizerFast.from_pretrained(path)

# Checkpoint.
checkpoint = torch.load(
    f"{path}/tinylm.pt",
    map_location="cpu",     # for compatibility.
)

# Model.
model = TinyLM(**checkpoint["config"])
model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()

TinyLM(
  (embedding): TinyEmbedding(
    (word_embedding): Embedding(8000, 128)
    (position_embedding): Embedding(256, 128)
    (emb_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (blocks): ModuleList(
    (0): TinyBlock(
      (attention): TinyAttention(
        (wq): Linear(in_features=128, out_features=128, bias=True)
        (wk): Linear(in_features=128, out_features=128, bias=True)
        (wv): Linear(in_features=128, out_features=128, bias=True)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (attn_out): Linear(in_features=128, out_features=128, bias=True)
        (attn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      )
      (attn_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
      (ffn): TinyFFN(
        (hidden1): Linear(in_features=128, out_features=256, bias=True)
        (hidden2): Linear(in_features=256, out_features=128, bias